# 😻 KittenTTS — Gradio Interface
**State-of-the-art TTS under 25MB · CPU-friendly · No GPU required**

Run each cell in order. The Gradio UI will launch at the bottom.

In [ ]:
# Cell 1: Install dependencies
!pip install -q https://github.com/KittenML/KittenTTS/releases/download/0.8.1/kittentts-0.8.1-py3-none-any.whl
!pip install -q gradio soundfile numpy librosa

In [ ]:
# Cell 2: Imports & model loader
import gradio as gr
import soundfile as sf
import numpy as np
import tempfile
from kittentts import KittenTTS

MODELS = {
    "kitten-tts-mini  (80M · best quality)": "KittenML/kitten-tts-mini-0.8",
    "kitten-tts-micro (40M · balanced)"    : "KittenML/kitten-tts-micro-0.8",
    "kitten-tts-nano  (15M · lightest)"    : "KittenML/kitten-tts-nano-0.8-fp32",
    "kitten-tts-nano  (15M · INT8 25MB)"   : "KittenML/kitten-tts-nano-0.8-int8",
}

VOICES = ['Bella', 'Jasper', 'Luna', 'Bruno', 'Rosie', 'Hugo', 'Kiki', 'Leo']
SAMPLE_RATE = 24000
_loaded = {}

def get_model(label):
    repo = MODELS[label]
    if repo not in _loaded:
        print(f'Loading {repo} ...')
        _loaded[repo] = KittenTTS(repo)
    return _loaded[repo]

print('Setup complete!')

In [ ]:
# Cell 3: Gradio UI
def synthesize(text, model_label, voice, speed):
    if not text.strip():
        return None, 'Please enter some text.'
    try:
        model = get_model(model_label)
        audio = model.generate(text, voice=voice)
        if speed != 1.0:
            import librosa
            audio = librosa.effects.time_stretch(audio.astype(np.float32), rate=speed)
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        sf.write(tmp.name, audio, SAMPLE_RATE)
        duration = len(audio) / SAMPLE_RATE
        status = f'Generated {duration:.1f}s of audio | Voice: {voice}'
        return tmp.name, status
    except Exception as e:
        return None, f'Error: {e}'

EXAMPLES = [
    'Welcome to KittenTTS, a lightweight text-to-speech model that runs entirely on CPU!',
    'The quick brown fox jumps over the lazy dog.',
    'Artificial intelligence is transforming the way we interact with technology.',
    'Hello! My name is Jasper, powered by a tiny 15-million-parameter neural network.',
]

css = '''
#header { text-align: center; padding: 20px 0 10px; }
#header h1 { font-size: 2.4em; margin-bottom: 4px; }
#header p  { color: #888; margin: 0; }
.gen-btn { background: linear-gradient(135deg,#a855f7,#ec4899) !important; color: white !important; }
'''

with gr.Blocks(title='KittenTTS', theme=gr.themes.Soft(primary_hue='purple', secondary_hue='pink'), css=css) as demo:
    gr.HTML("<div id='header'><h1>KittenTTS</h1><p>State-of-the-art TTS under 25MB &middot; No GPU needed</p></div>")

    with gr.Row():
        with gr.Column(scale=3):
            text_input = gr.Textbox(label='Text to synthesize', placeholder='Enter any text here...', lines=5)
            char_count = gr.Markdown('0 characters')
            text_input.change(fn=lambda t: f'{len(t)} characters', inputs=text_input, outputs=char_count)
            with gr.Row():
                voice_dd = gr.Dropdown(choices=VOICES, value='Jasper', label='Voice')
                model_dd = gr.Dropdown(choices=list(MODELS.keys()), value=list(MODELS.keys())[0], label='Model')
            speed_sl = gr.Slider(minimum=0.5, maximum=2.0, value=1.0, step=0.05, label='Speed')
            gen_btn  = gr.Button('Generate Speech', elem_classes='gen-btn', variant='primary')
            status   = gr.Markdown('')

        with gr.Column(scale=2):
            audio_out = gr.Audio(label='Output Audio', type='filepath', interactive=False)
            gr.Markdown('### Voice Guide\n| Voice | Style |\n|---|---|\n| Bella | Warm |\n| Jasper | Clear |\n| Luna | Calm |\n| Bruno | Deep |\n| Rosie | Bright |\n| Hugo | Rich |\n| Kiki | Cheerful |\n| Leo | Smooth |')

    gr.Markdown('### Example Texts')
    gr.Examples(
        examples=[[t, 'Jasper', list(MODELS.keys())[0], 1.0] for t in EXAMPLES],
        inputs=[text_input, voice_dd, model_dd, speed_sl],
    )

    gr.HTML("<hr><p style='text-align:center;color:#aaa;font-size:.85em'><a href='https://github.com/KittenML/KittenTTS' target='_blank'>GitHub</a> &middot; Apache-2.0</p>")

    gen_btn.click(fn=synthesize, inputs=[text_input, model_dd, voice_dd, speed_sl], outputs=[audio_out, status])

demo.launch(share=True)